# Predição de óbito ou readmissão em 6 meses em pacientes com insuficiência cardíaca

Trabalho final de Inteligência Computacional Aplicada a Problemas de Engenharia (UFMG 2026/2).

**Objetivo**: a partir das informações disponíveis na internação, prever se o paciente terá **óbito ou readmissão hospitalar em até 6 meses** (classificação binária).

Este *notebook* funciona como o **painel executivo** do projeto. Toda a lógica de processamento de dados e engenharia de atributos foi modularizada para os ficheiros Python `.py` na pasta `src/`.

Etapas:
0. Configurações Iniciais
1. Carregamento dos Dados
2. Análise exploratória dos dados (EDA)
3. Extração e seleção de atributos
4. Treinamento de modelos lineares e não lineares
5. Avaliação e seleção de modelos

## Sobre a base de dados

**Hospitalized patients with heart failure: integrating electronic healthcare records and external outcome data**, versão 1.3 (22/05/2022), PhysioNet.

- **Origem**: prontuários eletrônicos do *Zigong Fourth People's Hospital* (Sichuan, China), internações entre **dezembro/2016 e junho/2019**.
- **População**: **2008 pacientes** internados consecutivamente com insuficiência cardíaca (IC), diagnosticada pelos critérios da *European Society of Cardiology*.
- **Conteúdo** (`dat.csv`): dados demográficos e de admissão, sinais vitais, classificação da IC (tipo, NYHA, Killip), comorbidades (índice de Charlson), nível de consciência (Glasgow), ecocardiograma, ~100 exames laboratoriais e **desfechos de seguimento**.
- **Medicamentos** (`dat_md.csv`): fármacos administrados durante a internação (diuréticos, inotrópicos, vasodilatadores, etc.), em formato longo.

**Referência:** Zhang, Z., Cao, L., Zhao, Y., Xu, Z., Chen, R., Lv, L., & Xu, P. (2022). *Hospitalized patients with heart failure: integrating electronic healthcare records and external outcome data* (version 1.3). PhysioNet. https://doi.org/10.13026/5m60-vs44

## 0. Configurações Iniciais

Utilizamos o módulo `src/utils.py` para padronizar diretórios, variáveis globais e temas de plotagem (Matplotlib/Seaborn). 

In [ ]:
# Pedir ao IPython para recarregar módulos automaticamente
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Adiciona a raiz do repositório ao sys.path
BASE_DIR = Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

# Importações dos módulos customizados
from src import utils, preprocessing, features, eda

# Aplica o tema visual padronizado para gráficos
utils.setup_plotting()

## 1. Carregamento dos Dados

Os dados brutos são carregados através da função `load_data()` do módulo `preprocessing.py`. Ela obtém os dados clínicos (`dat.csv`), a tabela de medicamentos (`dat_md.csv`) e o dicionário de variáveis.

In [ ]:
patients, drugs, data_dictionary = preprocessing.load_data()

print(f"Base clínica (pacientes x variáveis): {patients.shape}")
print(f"Base de medicamentos (pacientes x medicamento): {drugs.shape}")
print(f"Dicionário das variáveis (variáveis x descrição): {data_dictionary.shape}")

## 2. Análise Exploratória de Dados

Visão geral dos dados:

In [ ]:
# A linha abaixo fere a política de privacidade do Dataset!!
# patients.head()

In [ ]:
patients.dtypes.value_counts()

### 2.1 Dicionário de Variáveis

As 166 colunas de `dat.csv` foram agrupadas por temas clínicos pelo dicionário de dados. O módulo `preprocessing.py` mapeia esses agrupamentos em `get_variable_groups()` para facilitar a visualização e filtragem (ex: remoção de dados de alta).

| Grupo | O que contém |
|---|---|
| Identificação | Número da internação |
| Demografia | Sexo, faixa etária (só por década) e ocupação |
| Admissão e histórico | Setor de admissão, se foi emergência e nº de internações anteriores |
| Sinais vitais e antropometria | Temperatura, pulso, respiração, pressão arterial, peso, altura e IMC na admissão |
| Classificação da IC | Tipo de IC (esquerda/direita/ambas), classe funcional NYHA e classe de Killip |
| Comorbidades | Doenças prévias (binárias) e o índice de comorbidade de Charlson (CCI) |
| Estado clínico na admissão | Insuficiência respiratória tipo II, consciência, componentes da escala de Glasgow, suporte ventilatório, oxigenoterapia, FiO2 e lesão renal aguda |
| Ecocardiograma | Fração de ejeção (LVEF), diâmetro do VE, velocidades das ondas mitrais E/A e refluxo tricúspide |
| Lab. — * | Exames laboratoriais, separados por painel |
| Desfechos e seguimento | Óbito/readmissão em 28 dias, 3 e 6 meses, desfecho na internação, retorno ao PS e tempos até cada evento |
| Informações da alta | Destino da alta, setor da alta e dias de internação |

<br>

> Lab. -*: aglutinação de 8 grupos diferentes, categorizados por tipos de exame laboratorial. (Total de 18 grupos).

Os grupos *Identificação*, *Desfechos e seguimento* e *Informações da alta* só são conhecidos após a alta (ou não são informação clínica) e serão removidos posteriormente.

In [ ]:
variable_groups = preprocessing.get_variable_groups(patients)
print(f"Foram definidos {len(variable_groups)} grupos de variáveis.")

### 2.1.1 Resumo Visual do Dicionário (Tabela 1)

Através do módulo `eda.py`, conseguimos gerar a tabela que mapeia todos os campos, a quantidade de dados em falta e os valores observados para cada variável clínica.

- **Tabelas detalhadas por grupo clínico:** As 18 tabelas detalhadas (variável, descrição, tipo, % de faltantes e valores observados) não são exibidas aqui. A função `display_variable_overview` exporta as tabelas para um arquivo PDF, com uma tabela por grupo, uma por página, e gera também um arquivo CSV com os dados brutos, salvando ambos na pasta `results`. 

In [ ]:
overview = eda.generate_variable_overview(patients, data_dictionary, variable_groups)
eda.display_variable_overview(overview)

### 2.2 Medicamentos (`dat_md.csv`)

A base de medicamentos encontra-se num **formato longo** (uma linha por par paciente-fármaco). Nesta versão, a informação disponível é apenas binária: **usou ou não usou** determinado fármaco, sem dosagem nem frequência.

In [ ]:
eda.generate_drug_summary(drugs, patients)

### 2.3 Valores Faltantes

Nem todo `NA` é um dado perdido aleatoriamente (Missing At Random). Exames como gasometria e perfil lipídico costumam faltar em blocos, indicando que **o painel não foi pedido**. Como essa ausência carrega informação clínica (o médico julgou desnecessário), utilizaremos indicadores de falta (*missing flags*) na etapa de Engenharia de Atributos em vez de apenas imputar os valores.

A função a seguir, do modulo `eda.py`, calcula o percentual exato de valores nulos (ausentes) para cada uma das 166 colunas da base de pacientes e gera um gráfico de barras com a porcentagem de faltantes.

> Filtragem: Para não poluir o gráfico, a função exclui as variáveis que estão 100% preenchidas e ordena as restantes da menor para a maior taxa de ausência.

In [ ]:
eda.plot_missing_rate_by_variable(patients)

**Quanto falta em cada variável**: 
- Cada barra é uma variável com algum faltante. As linhas tracejadas marcam 20% e 50%, limiares usuais na literatura para descartar uma variável em vez de imputá-la. Serve para escolher esse limiar posteriormente: há um grupo grande acima de 50% (gasometria, lipídios, ecocardiograma) e quase todo o resto abaixo de 20%.

**Interpretação para a modelagem**:
- Abaixo de 20%: Valores ausentes aqui (como em alguns exames básicos) costumam ser aleatórios e seguros para imputação (substituir o nulo pela mediana ou média da população).
- Acima de 50%: Barras longas que ultrapassam a segunda linha (como gasometria, lipoproteínas, ou respiratory.support. com 97,9% de ausência) indicam dados que não faltam ao acaso (Not Missing At Random). Tentar "adivinhar" o valor para 70% dos pacientes introduziria um viés grave. Em vez disso, a imagem justifica a adoção de Missing Flags (criar uma coluna binária dizendo "o exame não foi pedido"), pois a ausência carrega a informação clínica de que o médico não suspeitou daquela complicação.

### 2.3.1 Matriz de Valores Faltantes

Esta visualização comprova o padrão NMAR (Not Missing At Random). As faixas que atravessam o painel confirmam que não faltam pontos aleatórios de dados, mas sim painéis inteiros que não foram solicitados pelo médico.\
A função `plot_missing_matrix` gera um "mapa de calor" booleano que permite visualizar a estrutura e o comportamento dos dados ausentes em toda a base de pacientes.
O eixo horizontal representa as variáveis agrupadas, e o eixo vertical representa os 2.008 pacientes. Onde o dado está preenchido, a imagem fica na cor clara (fundo neutro); onde o dado é nulo/ausente, a imagem é preenchida por blocos azuis (cor da série).
- **Impacto na Modelagem**: Esse comportamento em bloco demonstra que a ausência ocorre porque o painel laboratorial inteiro não foi solicitado pelo médico. Como essa ausência indica uma decisão médica presuntiva (o médico julgou o paciente estável o suficiente para não precisar da gasometria, por exemplo), a imagem suporta metodologicamente a decisão de não imputar esses valores, mas sim transformá-los em atributos indicativos (Missing Flags).

In [ ]:
eda.plot_missing_matrix(patients, variable_groups)

### 2.4 Variável Alvo: Óbito ou Readmissão em 6 meses

**Motivação**: um teste preliminar comparou vários desfechos candidatos (readmissão em 28 dias/3/6 meses, óbito, alta a pedido, tempo de internação, classe NYHA). O composto *óbito ou readmissão em 6 meses* é o endpoint mais usado em estudos de prognóstico de IC, tem as classes mais equilibradas e evita uma distorção da readmissão pura: quem morre após a alta não pode ser readmitido e seria contado como "sem evento".

**Definição**: `target = 1` se o paciente foi readmitido em qualquer janela registrada (28 dias, 3 ou 6 meses) **ou** morreu em até 6 meses; senão `target = 0`. Usar qualquer janela de readmissão (e não só a de 6 meses) cobre 3 registros inconsistentes em que há readmissão em 3 meses mas não em 6 (ver checagem abaixo).

O gráfico a seguir mostra a prevalência de cada desfecho disponível na base, para situar o alvo escolhido em relação às alternativas: desfechos de óbito são raros demais (< 3%) para modelar com segurança, enquanto readmissão e o composto têm classes suficientemente equilibradas.

A função `create_target()` do módulo `preprocessing.py` realiza o agrupamento destas variáveis.

**Composição do alvo e checagens de consistência**: Os desfechos são cumulativos no tempo (quem teve o evento em 28 dias deve tê-lo também em 3 e 6 meses); óbito e readmissão em 6 meses não se sobrepõem nesta base. Os óbitos durante a própria internação também entram no alvo, pois estão contidos em `death.within.6.months`.

In [ ]:
target = preprocessing.create_target(patients)
print(f"Total no alvo: {target.sum()} de {len(target)} ({target.mean():.1%})")

# Gera a tabela de resumo, o gráfico de barras (prevalência) e o detalhamento
eda.plot_outcome_prevalence(patients, target)
eda.print_outcome_details(patients, target)

### 2.5 Análise Bivariada Clínica (Tabela 1 e Distribuições)

Para avaliar a relação individual entre as características basais da internação e o desfecho composto (`target`), realizamos uma análise bivariada estratificada entre os dois grupos populacionais: pacientes que permaneceram estáveis (`target = 0`) e pacientes que evoluíram para óbito ou readmissão em até 6 meses (`target = 1`).

Esta etapa cumpre dois objetivos metodológicos centrais:
1. **Caracterização Basal Estratificada (Tabela 1):** Apresenta a média e o desvio padrão das variáveis numéricas contínuas/discretas, bem como a distribuição percentual das variáveis categóricas críticas da insuficiência cardíaca (classe funcional NYHA, grau de Killip, faixa etária e sexo) em cada grupo.
2. **Avaliação de Separabilidade e Outliers:** Gera gráficos de caixa (*boxplots*) para marcadores clínicos e laboratoriais de referência (`LVEF`, `CCI.score`, pressão arterial sistólica e `brain.natriuretic.peptide`), permitindo inspecionar diferenças de mediana, dispersão interquartil e a presença de valores atípicos (*outliers*) biológicos antes da etapa de pré-processamento e normalização.

In [ ]:
# Seleção de características centrais para compor a tabela base do artigo
key_clinical_features = [
    'ageCat', 'gender', 'BMI', 'LVEF', 'CCI.score', 
    'systolic.blood.pressure', 'Killip.grade', 'NYHA.cardiac.function.classification'
]

# Gera a Tabela 1 estratificada
eda.generate_table1_stratified(patients, target, key_clinical_features)

# Plota os gráficos comparativos para identificação visual de outliers e separabilidade
eda.plot_clinical_bivariate(patients, target)

## 3. Extração e Engenharia de Atributos

A função `features.build_feature_set()` executa três passos críticos de forma limpa e modular:
1. **Remoção de Vazamento de Informação (Data Leakage)**: Remove desfechos temporais e informações exclusivas do momento da alta.
2. **Missing Flags**: Cria colunas indicadoras binárias (ex: `coluna_is_missing = 1`) para exames com mais de 20% de valores nulos.
3. **Pivotamento de Medicamentos**: Transforma a base `dat_md.csv` para o formato *wide*, em que cada fármaco se torna uma coluna binária acoplada à base clínica.

In [ ]:
# Orquestração do pipeline de limpeza e feature engineering
# X = features.build_feature_set(patients, drugs, variable_groups)

# print(f"Dimensão do conjunto de features final (X): {X.shape}")
# X.head()

# 1. Remoção de Vazamento de Informação (Data Leakage)
X = features.build_feature_set(patients, drugs, variable_groups)

print(f"\nDimensão original: {patients.shape}")
print(f"Dimensão da matriz preditora após remover vazamento: {X.shape}")

# 2. Tipologia explícita
variable_types = preprocessing.get_variable_types(X)
print("\nDistribuição da tipologia das variáveis preditoras:")
print(f"- Contínuas: {len(variable_types['continuas'])}")
print(f"- Discretas: {len(variable_types['discretas'])}")
print(f"- Binárias: {len(variable_types['binarias'])}")
print(f"- Categóricas: {len(variable_types['categoricas'])}")

## 4. Treinamento e Avaliação de Modelos

*(Esta secção será preenchida nas próximas fases do plano, importando os módulos `train.py` e `evaluate.py` para treinar os algoritmos e plotar as Curvas ROC e PR)*.

## Limpeza dos Resultados (Opcional)

Utilize (remove os comentários) para realizar a deletar os resultados produzidos, por qualquer que seja o motivo.
Caso deseje deletar os resultados para realizar uma nova execução e produção dos resultados, **lembre-se de comentar o código abaixo** para que ela não seja executado, o que resultaria na exclusão dos arquivos novamente.

In [ ]:
from src.utils import clean_results

# clean_results()